# GENE7 — supervised morphology contrasts: crispant vs matched control

**Question.** Within a stage- and temperature-matched stratum, is there a direction in morphology
space that separates a crispant target from its controls — and if so, does each embryo's *signed
distance* along that direction generalise well enough to use as a graded severity score?

**Why this and not the intra-cohort PCs.** The unsupervised cohort axes in `gene7_cohort_axes.ipynb`
found real, interpretable phenotype, but nothing forced them to point at the perturbation. This
notebook asks the supervised version. The resulting distance `s` is the intended replacement for the
binary crispant/control indicator in the PLN regressions.

**The premise being tested.** Crispants are mosaic F0s, so an injected clutch is a mixture of
effectively-null and escaper embryos. A binary label misclassifies the escapers, which attenuates
the contrast; a graded score partially recovers the dose. If that story is right, crispant groups
should be *more dispersed* along the discriminant than their controls.

**This notebook is homework, not the result.** It decides which contrasts have an axis worth
regressing on. No sequencing data is touched.

---

### Design

| | |
|---|---|
| space | first 5 axes of the GENE7-native 10-component PCA on `z_mu_b_*`, unwhitened |
| contrast | 3 targets × 4 temperatures × 3 timepoints = 36, controls matched within each cell |
| estimator | shrunken LDA (Ledoit–Wolf), equal priors |
| score | `s = (w·x + b) / ‖w‖`, signed, positive toward crispant |

Two caveats that hold throughout. Controls are **shared across the three targets** within a
condition, so the 36 contrasts are not independent. And every statistic here has a null that is not
zero: chance AUC is 0.5, and two arbitrary directions in 5D sit a median of ~70° apart, not 90°.


In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import numpy as np
import pandas as pd

HERE = Path.cwd()
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parents[2] / "src"))

import lda_contrasts as lc
import lda_plots as lp

lp.use_house_style()
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

DATA = HERE / "data" / "lda"
FIGURES = HERE / "figures" / "lda"

# --- shared figure/exclusion config (see gene7_config.py) ---
import gene7_config as cfg
FIGURES = cfg.figure_dir("lda_contrasts")
print(f"excluding {cfg.EXCLUDED_CONTRASTS} | figures -> {FIGURES}")


## 1. The shared coordinate system

Read from `data/gene7_global_scores.csv` rather than refit, so these contrasts live in exactly the
basis the cohort-PC covariates already use.


In [ ]:
scores = pd.read_csv(HERE / "data" / "gene7_global_scores.csv")
columns = lc.shared_columns(scores)
print(f"{len(scores)} GENE7 wells (curated image-QC exclusions already applied)")
print(f"shared subspace: {columns}")

design = (
    scores.pivot_table(index="perturbation_group", columns=["temperature", "timepoint_seq"],
                       values="well_id", aggfunc="count")
    .fillna(0).astype(int)
)
design


Every crispant cell has a control cell at the identical temperature and timepoint — the matching is
by design, not by modelling.


In [ ]:
axes = lc.fit_all_contrasts(scores, verbose=False)
summary = lc.contrast_summary(axes)
per_embryo = lc.contrast_scores(axes)
directions = lc.contrast_directions(axes)
similarity = lc.direction_similarity(axes)
print(f"{len(axes)} contrasts fitted")


## 2. Does the axis generalise?

The headline number is **leave-one-out AUC**: each embryo scored by a discriminant fit on the other
n−1. This is deliberately not in-sample separation. At 11-vs-12 in 5D the classes are nearly always
separable in-sample, so an in-sample statistic would be near-perfect everywhere and tell us nothing.
The p-values come from label permutation, re-running the full LOO procedure each time.


In [ ]:
figure = lp.auc_heatmap(summary)
lp.save_inline(figure, FIGURES / "00_auc_heatmap")


Read the `atf6` row against the others. Two of its cells sit at AUC 0.02 and 0.18 — *below* chance,
which is what an axis fit entirely to noise looks like: in-sample the classes separate, out-of-sample
the direction points backwards. That is the failure mode LOO exists to expose, and it is invisible to
Hotelling's T² or to any in-sample measure.

**Why several respectable-looking AUCs are not significant.** The null for this statistic is not
"0.5 plus a little noise". At ~11 per group, a *shuffled* labelling still reaches AUC 0.69–0.76 at
its 95th percentile, because shrunken LDA in 5D can partially fit any split of these embryos. The
column below is the bar each contrast actually had to clear.


In [ ]:
bar = summary.sort_values("loo_auc", ascending=False)[
    ["contrast", "n_crispant", "n_control", "loo_auc", "null_auc_p95", "perm_p_auc", "q_auc"]
].copy()
bar["clears_null_p95"] = bar["loo_auc"] > bar["null_auc_p95"]
bar.round(3).reset_index(drop=True).head(24)


In [ ]:
gates = pd.DataFrame({
    "permutation q<0.10 (LOO-AUC)": [(summary["q_auc"] < 0.10).sum()],
    "Hotelling p<0.05":             [(summary["hotelling_p"] < 0.05).sum()],
    "better determined than random":[(summary["boot_angle_p95"] < summary["random_angle_median"]).sum()],
    "usable (AUC gate AND stability gate)": [summary["usable"].sum()],
    "stage-confounded (angle<35°)": [summary["stage_confounded"].sum()],
}, index=[f"of {len(summary)}"]).T
gates


## 3. Where the embryos actually sit

One panel per condition, per target. The vertical dashed line is the hyperplane. This is the figure
that answers the dilution worry directly: if controls spread as widely along `s` as crispants do,
then a continuous score is carrying variation a binary label would have suppressed — and whether
that variation is signal or noise becomes an empirical question rather than an assumption.


In [ ]:
figure = lp.score_panels(per_embryo, summary, target="ctcf")
lp.save_inline(figure, FIGURES / "01_score_panels_ctcf")


In [ ]:
figure = lp.score_panels(per_embryo, summary, target="wfs1a,wfs1b")
lp.save_inline(figure, FIGURES / "02_score_panels_wfs1a-wfs1b")


In [ ]:
figure = lp.score_panels(per_embryo, summary, target="atf6")
lp.save_inline(figure, FIGURES / "03_score_panels_atf6")


## 4. The mosaic-F0 premise

If injected clutches mix effective nulls with escapers, they should be more dispersed than controls
along the axis that separates them. Because the discriminant is refit on every permuted labelling,
the accompanying null answers the obvious objection: *would an arbitrary split of these same embryos
show one side more spread out, once a discriminant is fit to it?*


In [ ]:
figure, dispersion = lp.dispersion_scatter(per_embryo, summary)
lp.save_inline(figure, FIGURES / "04_dispersion_scatter")


**Note the split verdict, because it matters for how this gets described.** No single contrast clears
its own permutation null — a variance ratio estimated from ~11 vs ~12 embryos simply has no power.
The evidence is entirely in the *consistency*: 29 of 36 contrasts point the same way, at a median
ratio of ~1.8×. That is a strong aggregate result and a null result per contrast, and it should be
reported as both.


In [ ]:
dispersion.sort_values("log_sd_ratio", ascending=False).head(10).round(3)


## 5. Is the direction determined, or arbitrary?

Separation and stability are different properties. A contrast can have significantly different
centroids and still recover a near-arbitrary direction under resampling — which would make `s`
meaningless for any embryo the axis was not built around. The grey band is where an arbitrary 5D
direction lands.


In [ ]:
figure = lp.stability_plot(axes, summary)
lp.save_inline(figure, FIGURES / "05_stability_plot")


## 5b. Consequential variability — does the *ordering* survive?

The angle above is a property of the estimator. What a regression on `s` actually consumes is the
**ordering** of embryos, so that is what should be measured: refit on each bootstrap, re-score every
original embryo, and Spearman-correlate against the full-data ordering.

The two can come apart, for a concrete geometric reason. The angle is measured in the ambient
Euclidean metric, but the embryos do not fill that space — within the 5D subspace, G00 alone carries
~49% of the variance. A direction perturbation lying mostly orthogonal to where the embryos actually
sit barely moves any score, so a 45° wobble can leave the ranking untouched.

**The floor is doubly important here.** Because the cloud is anisotropic, two *arbitrary* directions
already order these same embryos at ρ ≈ 0.47 with no information at all. So each contrast is tested
against its own random-direction null, with the observed bootstrap median compared against the full
null distribution — not one tail against the other.


In [ ]:
figure = lp.rank_stability_plot(axes, summary)
lp.save_inline(figure, FIGURES / "06_rank_stability_plot")


In [ ]:
figure = lp.angle_versus_rank(summary)
lp.save_inline(figure, FIGURES / "07_angle_versus_rank")


**Verdict: the ordinal measure is the better one to quote, but here it does not change the call.**

The left panel shows why the instinct was sound — the relationship is strong (Spearman −0.72) but
badly *nonlinear*. Below ~70° of angular wobble, ordinal stability is uniformly high (ρ ≈ 0.78–1.0)
and the angle barely discriminates within that range; past 70° it falls off a cliff. So the angle
saturates exactly where the working contrasts live, and reading a 45° spread as "half-arbitrary"
overstates the damage. The ordinal number says what it costs: essentially nothing.

Where it lands in practice: the ordinal gate passes **17/36** against the angular gate's **16/36**,
and the two agree on **35/36** contrasts. Different instrument, same reading — which is a useful
thing to know rather than a wasted check, because it means the earlier stability conclusions do not
need revisiting.

Top-tercile retention tells the same story in the units that matter for a severity score: 0.875
observed against a 0.625 floor, i.e. the severe end stays the severe end.


In [ ]:
best = summary.sort_values("loo_auc", ascending=False)["contrast"].iloc[0]
figure = lp.rank_churn_figure(next(a for a in axes if a.label == best))
lp.save_inline(figure, FIGURES / "08_rank_churn_figure")


In [ ]:
summary.sort_values("rank_rho_median", ascending=False)[
    ["contrast", "loo_auc", "boot_angle_p95", "rank_rho_median", "random_rank_rho_median",
     "rank_p_vs_floor", "q_rank", "top_retention_median", "loo_rank_rho",
     "usable", "usable_ordinal"]
].round(3).reset_index(drop=True)


## 6. Interpretation checks: what else is the axis parallel to?

**Stage.** Morphology encodes developmental stage strongly, and controls at three timepoints give
the stage direction directly, estimated from unperturbed embryos only. A discriminant near-parallel
to it is a delay score, and any transcriptional hit would be a stage-composition shift in disguise.

**The unsupervised cohort PC1.** Near 0° means the supervised axis recovers what the intra-cohort
PCA already found. Near the random median means the perturbation direction is a *minor* axis of
within-cohort variation — real phenotype dominating the cohort, but not the phenotype that separates
crispant from control.


In [ ]:
figure = lp.confound_scatter(summary)
lp.save_inline(figure, FIGURES / "09_confound_scatter")


In [ ]:
strong = summary.loc[summary["usable"]].sort_values("angle_to_stage")
strong[["contrast", "loo_auc", "q_auc", "angle_to_stage", "angle_to_cohort_pc1",
        "boot_angle_p95", "stage_confounded"]].round(3)


## 7. Do contrasts share a direction?

If a gene has a consistent morphological signature, its contrasts should point the same way across
temperatures and timepoints.

The floor here is easy to get wrong, so it is computed rather than asserted: in 5D the density of
`cos` between two random unit vectors goes as `(1 − c²)`, giving `E|cos| = 0.375`. Two *entirely
unrelated* directions already agree to 0.375, so a measured similarity of 0.4 is nothing.


In [ ]:
figure = lp.direction_heatmap(directions, summary)
lp.save_inline(figure, FIGURES / "10_direction_heatmap")


In [ ]:
figure = lp.similarity_heatmap(similarity)
lp.save_inline(figure, FIGURES / "11_similarity_heatmap")


In [ ]:
lc.direction_similarity_summary(
    axes, restrict_usable=summary.set_index("contrast")["usable"]
).round(4)


**A null result, and a clean one.** Restricted to contrasts with a usable axis, within-target
agreement is no higher than between-target agreement, and neither exceeds the random-direction floor.
So each contrast's discriminant is its own direction: `ctcf` at 24 °C and `ctcf` at 34 °C are not
finding the same morphological signature, any more than `ctcf` and `wfs1a/wfs1b` are.

This is the same non-commensurability that sank the intra-cohort PC approach, and it constrains what
comes next: coefficients can be *compared* across contrasts, but the predictor cannot be pooled, and
there is no "the atf6 morphology axis" to speak of.


## 8. What does the axis look like?

The check no statistic replaces. Every embryo in the contrast, ordered along `s`, controls in blue
and crispants in red. If the ordering tracked mounting angle or focus rather than biology, it would
be obvious here.

All 36 strips are written to `figures/lda/strips/`.


In [ ]:
from IPython.display import Image, display

showcase = (
    summary.loc[summary["usable"]].sort_values("loo_auc", ascending=False)["contrast"].head(4)
)
for label in showcase:
    axis = next(a for a in axes if a.label == label)
    strip = lc.contrast_image_strip(axis)
    row = summary.loc[summary["contrast"] == label].iloc[0]
    figure = lp.contrast_strip_figure(
        strip, title=f"{label}   LOO-AUC {row['loo_auc']:.2f}  (q={row['q_auc']:.3f})"
    )
    display(figure)
    import matplotlib.pyplot as plt; plt.close(figure)
lp.save_inline(figure, FIGURES / "12_contrast_strip_figure")


For contrast, a failing axis — the one at AUC 0.02. Controls and crispants interleave along the
ordering, and no severity gradient is visible.


In [ ]:
failing = summary.sort_values("loo_auc")["contrast"].iloc[0]
axis = next(a for a in axes if a.label == failing)
row = summary.loc[summary["contrast"] == failing].iloc[0]
figure = lp.contrast_strip_figure(
    lc.contrast_image_strip(axis),
    title=f"{failing}   LOO-AUC {row['loo_auc']:.2f}  (q={row['q_auc']:.3f})  — FAILING",
)
lp.save_inline(figure, FIGURES / "13_contrast_strip_figure_2")


## 9. Readout

The full table, sorted by how well the axis generalises.


In [ ]:
summary.sort_values("loo_auc", ascending=False)[
    ["contrast", "n_crispant", "n_control", "loo_auc", "perm_p_auc", "q_auc",
     "separation", "hotelling_p", "boot_angle_p95", "angle_to_stage",
     "log_sd_ratio", "shrinkage", "usable"]
].round(3).reset_index(drop=True)


In [ ]:
by_target = summary.groupby("target").agg(
    n=("contrast", "size"),
    usable=("usable", "sum"),
    median_auc=("loo_auc", "median"),
    median_log_sd_ratio=("log_sd_ratio", "median"),
    median_angle_to_stage=("angle_to_stage", "median"),
).round(3)
by_temperature = summary.groupby("temperature").agg(
    n=("contrast", "size"),
    usable=("usable", "sum"),
    median_auc=("loo_auc", "median"),
).round(3)
display(by_target, by_temperature)


### Where this leaves the regression plan

The score `s` is defensible as a regression predictor **only for the contrasts that clear both
gates**. For the rest, `s` is a direction fit to noise, and substituting it for the binary indicator
would cost power rather than gain it — the attenuation worry, realised.

Open items before any PLN fitting:

1. **Which arm of the comparison to fit.** Three one-predictor models with identical df — `~ binary`,
   `~ s`, and the hinge `~ pmax(s − c, 0)` — so raw log-likelihood is a fair head-to-head, with a
   sign test across contrasts.
2. **The hinge threshold `c`** should be the control mean or the hyperplane, not `min(s)` among
   crispants, which would hang the whole predictor on one embryo.
3. **Cosine between `β_s` and `β_binary`** per contrast. This is the geometric claim as a statistic:
   same direction, larger magnitude. High cosine with larger `|β|` means morphology sharpened the
   same vector rather than finding a different one.
4. **The stage-flagged contrasts** need a decision — exclude, or carry with the caveat.

None of that is run here.
